# nb_03_train - train, track, register

Two runs in the experiment `exp_delivery_promise`:

- **Run A - baseline.** scikit-learn `LogisticRegression` (one-hot categoricals + scaling). Logged for comparison, **not registered**.
- **Run B - LightGBM.** Native `lgb.train`, objective `binary`, so the Booster returns **probabilities**. Logged **and registered** as `mdl_delivery_promise`.

`MODEL_PROFILE` picks the LightGBM settings: `v1` (Lab 3 → version 1) or `v2` (Lab 7 → version 2, early stopping). Same holdout every time, so versions compare directly.

Everything is logged explicitly (params, metrics, plots) so learners can see exactly what lands in the experiment.

**Parameters.**

In [ ]:
MODEL_PROFILE = "v1"                     # "v1" | "v2"
EXPERIMENT_NAME = "exp_delivery_promise"
MODEL_NAME = "mdl_delivery_promise"

**Imports.** Fabric autologging is switched off so the run contains only what this notebook logs on purpose.

In [ ]:
import pandas as pd, numpy as np
import lightgbm as lgb
import mlflow, mlflow.lightgbm, mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

mlflow.autolog(disable=True)
mlflow.set_experiment(EXPERIMENT_NAME)

# The 16 model inputs, in contract order (kit/contracts/feature_contract.json). Order matters: requests use orientation "values".
FEATURES = ["carrier_id", "ship_method_express", "warehouse_id", "region_id", "remoteness", "distance_km", "order_hour",
            "order_dow", "is_weekend", "is_peak", "items_count", "total_weight_kg", "order_value", "promised_days",
            "backlog_index", "forecast_severity"]
FLOAT_COLS = ["distance_km", "total_weight_kg", "order_value", "backlog_index"]   # double
INT_COLS = [c for c in FEATURES if c not in FLOAT_COLS]                           # long

CATEGORICAL = ["carrier_id", "warehouse_id", "region_id"]
assert MODEL_PROFILE in ("v1", "v2")

**Load the gold training table** and split it: train (before 2026-07-01) and holdout (on/after). The holdout is the exam the model has never seen.

In [ ]:
gold = spark.read.table("gold_training_delivery").toPandas()
gold = gold.sort_values("order_ts").reset_index(drop=True)
train = gold[gold.split == "train"]; hold = gold[gold.split == "holdout"]

def xy(df):
    X = df[FEATURES].copy()
    for c in INT_COLS: X[c] = X[c].astype("int64")      # long
    for c in FLOAT_COLS: X[c] = X[c].astype("float64")  # double
    return X, df["is_late"].astype(int).values

X_train, y_train = xy(train); X_hold, y_hold = xy(hold)
print(f"train {len(X_train)} rows (late {y_train.mean():.1%})   holdout {len(X_hold)} rows (late {y_hold.mean():.1%})")

**Metrics and a calibration plot**, shared by both runs.

In [ ]:
def metrics(y, p):
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "log_loss": log_loss(y, p), "brier": brier_score_loss(y, p)}

def calibration_figure(y, p, title):
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect")
    ax.plot(mean_pred, frac_pos, marker="o", label="model")
    ax.set_xlabel("mean predicted late_risk"); ax.set_ylabel("actual late rate"); ax.set_title(title); ax.legend()
    return fig

**Run A - logistic-regression baseline.** One-hot for the three ID columns, standard scaling for the rest. Logged, not registered.

In [ ]:
numeric = [c for c in FEATURES if c not in CATEGORICAL]
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
                         ("num", StandardScaler(), numeric)])
baseline = Pipeline([("prep", pre), ("lr", LogisticRegression(max_iter=2000, C=1.0))])

with mlflow.start_run(run_name="A_logreg_baseline") as run_a:
    mlflow.log_params({"model": "LogisticRegression", "C": 1.0, "encoding": "one-hot + StandardScaler",
                       "holdout_from": "2026-07-01", "n_train": len(X_train), "n_holdout": len(X_hold)})
    baseline.fit(X_train, y_train)
    p_a = baseline.predict_proba(X_hold)[:, 1]
    m_a = metrics(y_hold, p_a)
    mlflow.log_metrics({f"holdout_{k}": v for k, v in m_a.items()})
    mlflow.log_figure(calibration_figure(y_hold, p_a, "Baseline calibration (holdout)"), "calibration_holdout.png")
    mlflow.sklearn.log_model(baseline, artifact_path="model")   # logged for reference only; NOT registered
print({k: round(v, 4) for k, v in m_a.items()})

**LightGBM profiles.** v1 is small and fixed; v2 is larger and stops early on a validation slice (the last 15% of train by time).

In [ ]:
PROFILES = {
    "v1": {"num_leaves": 15, "learning_rate": 0.05, "num_boost_round": 200, "early_stopping": None},
    "v2": {"num_leaves": 31, "learning_rate": 0.05, "num_boost_round": 600, "early_stopping": 30},
}
prof = PROFILES[MODEL_PROFILE]
params = {"objective": "binary", "metric": "auc", "num_leaves": prof["num_leaves"], "learning_rate": prof["learning_rate"],
          "feature_fraction": 0.9, "bagging_fraction": 0.9, "bagging_freq": 1, "min_data_in_leaf": 50,
          "seed": 42, "verbose": -1}

if prof["early_stopping"]:
    cut = int(len(X_train) * 0.85)                     # time-ordered: first 85% fit, last 15% validate
    X_fit, y_fit, X_val, y_val = X_train.iloc[:cut], y_train[:cut], X_train.iloc[cut:], y_train[cut:]
else:
    X_fit, y_fit, X_val, y_val = X_train, y_train, None, None
print(MODEL_PROFILE, params, "fit rows:", len(X_fit), "val rows:", 0 if X_val is None else len(X_val))

**Run B - train the Booster** with the three ID columns declared categorical, then score the holdout.

In [ ]:
with mlflow.start_run(run_name=f"B_lightgbm_{MODEL_PROFILE}") as run_b:
    mlflow.log_params({**params, "profile": MODEL_PROFILE, "num_boost_round_max": prof["num_boost_round"],
                       "early_stopping_rounds": prof["early_stopping"], "categorical_feature": ",".join(CATEGORICAL),
                       "holdout_from": "2026-07-01", "n_train": len(X_train), "n_holdout": len(X_hold)})
    dtrain = lgb.Dataset(X_fit, label=y_fit, categorical_feature=CATEGORICAL, free_raw_data=False)
    valid_sets, callbacks = [dtrain], [lgb.log_evaluation(0)]
    if X_val is not None:
        valid_sets.append(lgb.Dataset(X_val, label=y_val, categorical_feature=CATEGORICAL, reference=dtrain))
        callbacks.append(lgb.early_stopping(prof["early_stopping"], verbose=False))
    booster = lgb.train(params, dtrain, num_boost_round=prof["num_boost_round"], valid_sets=valid_sets,
                        valid_names=["fit", "val"][:len(valid_sets)], callbacks=callbacks)
    mlflow.log_param("best_iteration", booster.best_iteration or booster.current_iteration())

    p_b = booster.predict(X_hold, num_iteration=booster.best_iteration or None)
    m_b = metrics(y_hold, p_b)
    mlflow.log_metrics({f"holdout_{k}": v for k, v in m_b.items()})
    mlflow.log_metric("holdout_auc_gap_vs_baseline", m_b["roc_auc"] - m_a["roc_auc"])
    mlflow.log_figure(calibration_figure(y_hold, p_b, f"LightGBM {MODEL_PROFILE} calibration (holdout)"), "calibration_holdout.png")

    # feature importance as a table and a chart
    imp = pd.DataFrame({"feature": booster.feature_name(), "gain": booster.feature_importance("gain"),
                        "splits": booster.feature_importance("split")}).sort_values("gain", ascending=False)
    imp.to_csv("feature_importance.csv", index=False); mlflow.log_artifact("feature_importance.csv")
    fig, ax = plt.subplots(figsize=(6, 5)); ax.barh(imp.feature[::-1], imp.gain[::-1]); ax.set_title("Feature importance (gain)")
    mlflow.log_figure(fig, "feature_importance.png")

    # signature: long/double columns in, one double out named late_risk. Cast first so the schema is column-based.
    X_sample = X_hold.head(200).copy()
    for c in INT_COLS: X_sample[c] = X_sample[c].astype("int64")
    for c in FLOAT_COLS: X_sample[c] = X_sample[c].astype("float64")
    p_sample = booster.predict(X_sample, num_iteration=booster.best_iteration or None)
    signature = infer_signature(X_sample, pd.Series(p_sample, name="late_risk"))

    model_info = mlflow.lightgbm.log_model(booster, artifact_path="model", signature=signature,
                                           input_example=X_sample.head(3), registered_model_name=MODEL_NAME)
    run_b_id = run_b.info.run_id

print(f"registered {MODEL_NAME} from run {run_b_id}")
print(signature)

**Score the holdout and write `gold_holdout_scored`** (order_id, late_risk, is_late, risk_decile). This table feeds the calibration page in Power BI.

In [ ]:
scored = pd.DataFrame({"order_id": hold["order_id"].values, "late_risk": p_b.astype("float64"), "is_late": y_hold.astype("int64")})
scored["risk_decile"] = (pd.qcut(scored["late_risk"].rank(method="first"), 10, labels=False) + 1).astype("int64")  # 1 = lowest risk
scored["model_profile"] = MODEL_PROFILE
(spark.createDataFrame(scored).write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable("gold_holdout_scored"))
print(scored.groupby("risk_decile").agg(avg_pred=("late_risk", "mean"), actual=("is_late", "mean"), n=("order_id", "size")).round(3))

**Check cell (Lab 3 checkpoint).** LightGBM holdout AUC must be ≥ 0.80 and beat the baseline by ≥ 0.03.

In [ ]:
gap = m_b["roc_auc"] - m_a["roc_auc"]
print(f"baseline  holdout ROC-AUC: {m_a['roc_auc']:.4f}   PR-AUC {m_a['pr_auc']:.4f}   log_loss {m_a['log_loss']:.4f}   brier {m_a['brier']:.4f}")
print(f"LightGBM  holdout ROC-AUC: {m_b['roc_auc']:.4f}   PR-AUC {m_b['pr_auc']:.4f}   log_loss {m_b['log_loss']:.4f}   brier {m_b['brier']:.4f}")
print(f"gap: {gap:+.4f}   ->", "PASS" if m_b["roc_auc"] >= 0.80 and gap >= 0.03 else "CHECK: outside the kit targets")
print("Now open", EXPERIMENT_NAME, "to compare the runs, and", MODEL_NAME, "to see the new version and its 16-column input schema.")